# COCOMO-Extended — Full data-collection run

Builds the study dataset: about **200 repositories that use AI coding agents and 400 matched comparison repositories** (2 per AI repository), measured in **quarterly windows from 2021 Q1 to 2026 Q2**.

For every repository-window it records effort (full-time-equivalent developer-months, Robles et al. 2022), size (KSLOC added, bulk commits excluded) and the six drivers (DVOP, AIAD, MSVC, CLAU, MTEX, SECU) as continuous 0–1 scores, plus robustness variants of effort.

**This takes several Colab sessions.** Each session processes repositories until its time budget runs out, saving each one as it finishes. Just run the notebook again (Runtime → Run all) and it continues where it stopped. Keep the browser tab open while it runs: Colab disconnects idle sessions.

**Use Google Drive.** Without Drive, results are lost when a session ends unless you download them (section 10) and upload them again at the start of the next session (section 1b).

Before starting: add the GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon on the left, notebook access on).

## 1. Setup

In [ ]:
!pip -q install huggingface_hub pyarrow fsspec statsmodels
USE_DRIVE = True
SAVE_ROOT = "/content/cocomo_extended"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_ROOT = "/content/drive/MyDrive/cocomo_extended"
    except Exception as e:
        print("Google Drive could not be mounted:", e)
        print("Continuing WITHOUT Drive: download the results at the end of every session (section 10).")
print("Results folder:", SAVE_ROOT)

### 1b. Only without Google Drive: restore earlier results
If you downloaded a results zip in an earlier session, set `RESTORE = True`, run this cell and choose the zip. Skip it otherwise.

In [ ]:
RESTORE = False
if RESTORE:
    import zipfile, io, os
    from google.colab import files
    up = files.upload()
    for name, data in up.items():
        os.makedirs(SAVE_ROOT + "/full_run", exist_ok=True)
        zipfile.ZipFile(io.BytesIO(data)).extractall(SAVE_ROOT + "/full_run")
        print("Restored", name)

In [ ]:
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    from getpass import getpass
    GITHUB_TOKEN = getpass('Paste your GitHub token: ')
print('Token loaded:', bool(GITHUB_TOKEN))

## 2. Settings
These values are the study's pre-registered rules. **Do not change them after the full run has started**, or the repositories already processed will have been measured differently.

* Effort: a developer with at least `THETA_6M` commits per 6 months counts as full-time; others count in proportion (Robles et al. 2022; 12 = OpenStack). The threshold is scaled to the window length. `THETA_ROBUST` and the active-days and 6-month versions are robustness checks.
* Drivers are kept as continuous 0–1 scores for the main model. `LEVEL_CUTS` only converts them to COCOMO levels for reporting.

In [ ]:
CFG = dict(
    START="2021-01-01", END="2026-07-01", WINDOW_MONTHS=3,
    # effort (Robles et al. 2022)
    THETA_6M=12, THETA_ROBUST=[17, 24],
    # inclusion rules for a window
    CORE_DEV_COMMITS=5, MIN_CORE_DEVS=3, MIN_KSLOC=1.0, BULK_COMMIT_LINES=2000,
    # repository screen
    MIN_STARS=100, LANGS=["Python", "JavaScript", "TypeScript", "Java", "Go", "C#", "Rust"],
    CREATED_BEFORE="2022-01-01", ACTIVE_AFTER="2026-05-01", MAX_REPO_MB=1500,
    ACTIVE_SINCE="2025-01-01", MIN_RECENT_COMMITS=300, MIN_CONTRIBUTORS=20,
    # sample
    N_AI=200, CONTROLS_PER_AI=2, MIN_AGENT_PRS=5, ADOPTED_BY="2025-09-30",
    MAX_CANDIDATES_CHECKED=3000, SEED=42,
    # value at which each indicator counts as fully 'on' (score = 1)
    FULL_SCORE_AT=dict(n_ci_workflows=5, ci_runs_per_week=20, releases_per_month=2, agent_pr_share=0.30,
                       ai_commit_share=0.30, n_services=50, iac_change_share=0.10, tenure_months=72),
    # reporting only: score -> COCOMO level (<0.10 VL, <0.30 L, <0.50 N, <0.70 H, <0.90 VH, else XH)
    LEVEL_CUTS=[0.10, 0.30, 0.50, 0.70, 0.90],
    # session control
    MAX_HOURS_PER_SESSION=10,
    OUT_DIR=SAVE_ROOT + "/full_run", CLONE_DIR="/content/clones", AIDEV_VERSION="v5",
)
import os, json
os.makedirs(CFG["OUT_DIR"] + "/repos", exist_ok=True)
os.makedirs(CFG["CLONE_DIR"], exist_ok=True)
# record the settings used, and refuse to continue with different ones
cfg_file = CFG["OUT_DIR"] + "/settings.json"
saved = json.load(open(cfg_file)) if os.path.exists(cfg_file) else None
current = {k: v for k, v in CFG.items() if k not in ("OUT_DIR", "CLONE_DIR", "MAX_HOURS_PER_SESSION")}
if saved is None:
    json.dump(current, open(cfg_file, "w"), indent=1)
    print("Settings saved.")
elif saved != current:
    raise SystemExit("Settings differ from the ones this run started with: " +
                     str({k: (saved.get(k), current.get(k)) for k in current if saved.get(k) != current.get(k)}))
else:
    print("Settings match the saved run.")

## 3. Measurement functions

In [ ]:
# ===================== MEASUREMENT FUNCTIONS (git-based) =====================
# Everything in this cell works from a local git clone; no API calls.
import os, re, math, json, shutil, subprocess
from collections import defaultdict
import pandas as pd
import numpy as np
import yaml

# ---------- windows ----------
def make_windows(start, end, months):
    """Return list of (label, start_ts, end_ts) half-open windows [start, end)."""
    edges = pd.date_range(start=start, end=end, freq=f"{months}MS", tz="UTC")
    out = []
    for a, b in zip(edges[:-1], edges[1:]):
        if months == 6:
            lab = f"{a.year}H{1 if a.month <= 6 else 2}"
        elif months == 3:
            lab = f"{a.year}Q{(a.month - 1) // 3 + 1}"
        else:
            lab = a.strftime("%Y-%m")
        out.append((lab, a, b))
    return out

# ---------- git helpers ----------
def run(cmd, cwd=None, timeout=3600):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                       errors="replace", timeout=timeout)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])}... failed: {r.stderr[:300]}")
    return r.stdout

def clone_repo(full_name, dest):
    if os.path.exists(dest):
        shutil.rmtree(dest)
    run(["git", "clone", "--quiet", "--no-checkout", "--single-branch",
         f"https://github.com/{full_name}.git", dest], timeout=5400)

REC, FLD = "\x1e", "\x1f"

def _clean_path(p):
    p = p.strip()
    if "{" in p and "=>" in p:
        p = re.sub(r"\{([^{}]*) => ([^{}]*)\}", r"\2", p)
        p = p.replace("//", "/")
    elif " => " in p:
        p = p.split(" => ")[1]
    return p

def read_commits(repo_dir, since, until):
    """Non-merge commits on the default branch with numstat, between since and until."""
    fmt = f"{REC}%H{FLD}%ae{FLD}%an{FLD}%aI{FLD}%B{FLD}"
    out = run(["git", "log", "--no-merges", "--numstat", f"--format={fmt}",
               f"--since={since}", f"--until={until}"], cwd=repo_dir)
    commits = []
    for rec in out.split(REC)[1:]:
        parts = rec.split(FLD)
        if len(parts) < 6:
            continue
        sha, email, name, date, body, stat = parts[0], parts[1], parts[2], parts[3], parts[4], parts[5]
        files = []
        for line in stat.strip().splitlines():
            cols = line.split("\t")
            if len(cols) != 3:
                continue
            a, d, p = cols
            files.append((int(a) if a.isdigit() else 0, int(d) if d.isdigit() else 0, _clean_path(p)))
        commits.append({"sha": sha, "email": email.lower().strip(), "name": name.strip(),
                        "date": pd.Timestamp(date).tz_convert("UTC"), "body": body, "files": files})
    return commits

def first_commit_dates(repo_dir):
    """First-ever commit date per author email (full history) for team-experience."""
    out = run(["git", "log", "--no-merges", "--format=%ae%x1f%aI", "--reverse"], cwd=repo_dir)
    first = {}
    for line in out.splitlines():
        if FLD not in line:
            continue
        e, d = line.split(FLD)
        e = e.lower().strip()
        if e not in first:
            first[e] = pd.Timestamp(d).tz_convert("UTC")
    return first

def tag_dates(repo_dir):
    out = run(["git", "for-each-ref", "refs/tags", "--format=%(creatordate:iso-strict)"], cwd=repo_dir)
    ds = []
    for line in out.splitlines():
        line = line.strip()
        if line:
            try:
                ds.append(pd.Timestamp(line).tz_convert("UTC"))
            except Exception:
                pass
    return ds

def commit_at(repo_dir, before_ts):
    out = run(["git", "rev-list", "-1", f"--before={before_ts.isoformat()}", "HEAD"], cwd=repo_dir).strip()
    return out or None

def list_files(repo_dir, sha):
    return run(["git", "ls-tree", "-r", "--name-only", sha], cwd=repo_dir).splitlines()

def file_size(repo_dir, sha, path):
    try:
        return int(run(["git", "cat-file", "-s", f"{sha}:{path}"], cwd=repo_dir).strip())
    except Exception:
        return 10**9

def read_file(repo_dir, sha, path, max_bytes=300_000):
    if file_size(repo_dir, sha, path) > max_bytes:
        return ""
    try:
        return run(["git", "show", f"{sha}:{path}"], cwd=repo_dir, timeout=120)
    except Exception:
        return ""

# ---------- classification of authors ----------
BOT_RE = re.compile(r"\[bot\]|(^|[^a-z])bot@|dependabot|renovate|github-actions|pre-commit-ci|codecov|"
                    r"greenkeeper|snyk-bot|allcontributors|semantic-release|mergify|imgbot|"
                    r"weblate|transifex|crowdin|azure-pipelines|travis", re.I)
AI_RE = re.compile(r"copilot|devin-ai|noreply@anthropic\.com|(^|[^a-z])claude($|[^a-z])|"
                   r"chatgpt-codex|openai-codex|codex\[bot\]|cursoragent|cursor\[bot\]|"
                   r"jules\[bot\]|openhands|aider|factory-droid|gemini-code-assist|sweep-ai", re.I)
COAUTHOR_RE = re.compile(r"^co-authored-by:\s*(.+)$", re.I | re.M)

def author_kind(name, email):
    s = f"{name} <{email}>"
    if AI_RE.search(s):
        return "ai"
    if BOT_RE.search(s):
        return "bot"
    return "human"

def has_ai_coauthor(body):
    return any(AI_RE.search(m) for m in COAUTHOR_RE.findall(body or ""))

# ---------- file classification ----------
SRC_EXT = {".py", ".js", ".jsx", ".ts", ".tsx", ".mjs", ".cjs", ".java", ".kt", ".scala", ".go",
           ".cs", ".rs", ".rb", ".php", ".c", ".cc", ".cpp", ".cxx", ".h", ".hpp", ".swift",
           ".m", ".vue", ".svelte", ".dart", ".ex", ".exs", ".clj", ".fs", ".lua"}
EXCL_DIR_RE = re.compile(r"(^|/)(vendor|third_party|thirdparty|node_modules|dist|build|out|target|"
                         r"generated|gen|\.yarn|bower_components|site-packages|external)/", re.I)
EXCL_FILE_RE = re.compile(r"(\.min\.js$|_pb2(_grpc)?\.py$|\.pb\.go$|\.pb\.(h|cc)$|\.g\.dart$|"
                          r"\.generated\.|\.designer\.cs$|bundle\.js$)", re.I)
TEST_RE = re.compile(r"(^|/)(tests?|__tests__|spec|specs|testing|e2e|test_?data|fixtures)/|"
                     r"(^|/)test_[^/]*$|_test\.[a-z]+$|\.(test|spec)\.[a-z]+$|Tests?\.(java|kt|cs)$", re.I)

def file_class(path):
    """'src', 'test' or None (not counted in size)."""
    ext = os.path.splitext(path)[1].lower()
    if ext not in SRC_EXT or EXCL_DIR_RE.search(path) or EXCL_FILE_RE.search(path):
        return None
    return "test" if TEST_RE.search(path) else "src"

CI_RE = re.compile(r"^(\.github/workflows/[^/]+\.ya?ml|\.gitlab-ci\.yml|\.circleci/config\.ya?ml|"
                   r"Jenkinsfile|\.travis\.yml|azure-pipelines\.ya?ml|\.drone\.yml|bitbucket-pipelines\.yml|"
                   r"\.buildkite/.+\.ya?ml)$")
# deployment folders: only at the top level (or one level under deploy-type roots), never inside source code
DEPLOY_ROOT = (r"(ops|tf|cfn|\.?[^/]*(deploy|k8s|kube|helm|chart|kustomiz|infra|terraform|manifest|istio|argocd|"
               r"cloudformation|pulumi|ansible|gitops)[^/]*)")
IAC_RE = re.compile(r"(\.tf$|\.tfvars$|\.bicep$|(^|/)Pulumi\.ya?ml$|(^|/)Chart\.ya?ml$|"
                    r"(^|/)kustomization\.ya?ml$|(^|/)serverless\.ya?ml$|(^|/)cdk\.json$|"
                    r"(^|/)skaffold\.ya?ml$|\.ya?ml$|\.json$)", re.I)
IAC_LOC_RE = re.compile(r"^(" + DEPLOY_ROOT + r"/|[^/]+/" + DEPLOY_ROOT + r"/)", re.I)
IAC_STRONG_RE = re.compile(r"(\.tf$|\.bicep$|(^|/)Pulumi\.ya?ml$|(^|/)cdk\.json$|(^|/)serverless\.ya?ml$|"
                           r"^skaffold\.ya?ml$|^cloudbuild\.ya?ml$|(^|/)Chart\.ya?ml$|(^|/)kustomization\.ya?ml$)", re.I)

def is_iac(path):
    """IaC file in a top-level deployment folder; Terraform/Bicep/Pulumi/CDK/Serverless files anywhere
    outside source, test, example and docs folders."""
    if NONPROD_DIR_RE.search(path):
        return False
    if IAC_STRONG_RE.search(path):
        return True
    return bool(IAC_LOC_RE.search(path)) and path.lower().endswith((".yml", ".yaml", ".json", ".tf", ".tfvars"))

COMPOSE_RE = re.compile(r"(^|/)(docker-)?compose[^/]*\.ya?ml$", re.I)
K8S_DIR_RE = IAC_LOC_RE   # Kubernetes manifests counted only in top-level deployment folders
DOCKERFILE_RE = re.compile(r"(^|/)Dockerfile[^/]*$|\.Dockerfile$", re.I)
NONPROD_DIR_RE = re.compile(r"(^|/)(tests?|testing|testdata|examples?|samples?|docs?|\.devcontainer|benchmarks?|demos?|e2e|"
                            r"fixtures|integration)/", re.I)
PACKAGING_DIR_RE = re.compile(r"(^|/)(dev|docker|ci|scripts?|tools|build|\.github|hack|contrib)(/|$)", re.I)
BUILD_FILE_RE = re.compile(r"(^|/)(package\.json|go\.mod|pyproject\.toml|setup\.py|requirements\.txt|pom\.xml|"
                           r"build\.gradle(\.kts)?|Cargo\.toml|[^/]+\.csproj|Gemfile|composer\.json|mix\.exs)$", re.I)
AI_CFG_RE = re.compile(r"^(\.github/copilot-instructions\.md|\.github/instructions/.+|\.cursorrules|"
                       r"\.cursor/.+|CLAUDE\.md|\.claude/.+|AGENTS\.md|GEMINI\.md|\.aider[^/]*|"
                       r"\.windsurfrules|\.clinerules.*|\.continue/.+)$", re.I)
SECPOL_RE = re.compile(r"^(\.github/|docs/)?SECURITY\.(md|rst|txt)$", re.I)
DEPUPD_RE = re.compile(r"^(\.github/dependabot\.ya?ml|renovate\.json5?|\.renovaterc(\.json)?|"
                       r"\.github/renovate\.json5?)$", re.I)
CODEOWNERS_RE = re.compile(r"^(\.github/|docs/)?CODEOWNERS$")
SAST_RE = re.compile(r"codeql|semgrep|snyk|bandit|gosec|sonar|trivy|grype|brakeman|"
                     r"ossf/scorecard|osv-scanner|zizmor", re.I)
DEPLOY_RE = re.compile(r"terraform\s+(apply|plan)|helm\s+(upgrade|install)|kubectl\s+(apply|rollout|set)|"
                       r"aws\s+(cloudformation|ecs|lambda|s3\s+sync)|gcloud\s+\S*\s*deploy|"
                       r"az\s+(webapp|deployment|containerapp)|serverless\s+deploy|pulumi\s+up|"
                       r"flyctl\s+deploy|vercel|netlify\s+deploy|docker/build-push-action|"
                       r"docker\s+push|environment:\s*(production|prod)\b", re.I)
TEST_STEP_RE = re.compile(r"pytest|tox\b|nox\b|npm\s+(run\s+)?test|yarn\s+(run\s+)?test|pnpm\s+(run\s+)?test|"
                          r"jest|vitest|mocha|go\s+test|cargo\s+(test|nextest)|mvn\s|gradle|dotnet\s+test|"
                          r"make\s+(test|check)|phpunit|rspec|ctest|playwright|cypress", re.I)
K8S_KIND_RE = re.compile(r"^kind:\s*(Deployment|StatefulSet|DaemonSet)\s*$", re.M)

# ---------- scoring ----------
def cap(x, full):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return np.nan
    return float(min(1.0, max(0.0, x / full)))

def mean_avail(xs):
    xs = [x for x in xs if x is not None and not (isinstance(x, float) and math.isnan(x))]
    return float(np.mean(xs)) if xs else np.nan

def to_level(score, cuts):
    """Map a 0-1 score to COCOMO levels coded VL=-2 ... XH=3."""
    if score is None or (isinstance(score, float) and math.isnan(score)):
        return np.nan
    return -2 + int(np.searchsorted(cuts, score, side="right"))

# ---------- tree-based measures at the end of a window ----------
def tree_measures(repo_dir, sha, max_yaml=300):
    files = list_files(repo_dir, sha)
    m = {}
    ci_files = [f for f in files if CI_RE.search(f)]
    m["has_ci"] = int(bool(ci_files))
    iac_files = [f for f in files if is_iac(f)]
    m["n_iac_files"] = len(iac_files)
    m["has_ai_config"] = int(any(AI_CFG_RE.search(f) for f in files))
    m["has_security_policy"] = int(any(SECPOL_RE.search(f) for f in files))
    m["has_dep_update"] = int(any(DEPUPD_RE.search(f) for f in files))
    m["has_codeowners"] = int(any(CODEOWNERS_RE.search(f) for f in files))

    # CI workflow contents: SAST and deployment steps
    ci_text = "\n".join(read_file(repo_dir, sha, f) for f in ci_files[:60])
    m["has_sast"] = int(bool(SAST_RE.search(ci_text)))
    m["ci_deploy"] = int(bool(DEPLOY_RE.search(ci_text)))
    m["has_test_in_ci"] = int(bool(TEST_STEP_RE.search(ci_text)))
    m["n_ci_workflows"] = len(ci_files)

    # services: docker-compose services with a build key
    compose_built = 0
    for f in [f for f in files if COMPOSE_RE.search(f) and not NONPROD_DIR_RE.search(f)][:20]:
        try:
            doc = yaml.safe_load(read_file(repo_dir, sha, f)) or {}
            svcs = doc.get("services", {}) if isinstance(doc, dict) else {}
            if isinstance(svcs, dict):
                compose_built = max(compose_built, sum(1 for v in svcs.values()
                                                       if isinstance(v, dict) and "build" in v))
        except Exception:
            pass
    # Kubernetes workloads in deployment directories
    # unique workload names, so the same service defined in Helm, Kustomize and plain manifests counts once
    names = set()
    yfiles = [f for f in files if f.lower().endswith((".yml", ".yaml")) and K8S_DIR_RE.search(f)
              and not NONPROD_DIR_RE.search(f)][:max_yaml]
    for f in yfiles:
        for doc in re.split(r"(?m)^---\s*$", read_file(repo_dir, sha, f)):
            if K8S_KIND_RE.search(doc):
                mm = re.search(r"(?m)^metadata:\s*\n(?:[ \t]+.*\n)*?[ \t]+name:\s*['\"]?([^\s'\"]+)", doc)
                nm = mm.group(1) if mm else f
                if "{{" in nm:                      # Helm template name: use the file name instead
                    nm = os.path.splitext(os.path.basename(f))[0]
                names.add(nm.lower())
    k8s = len(names)
    # directories with their own Dockerfile
    # a service = a folder with its own Dockerfile AND its own build file (not a packaging/dev image folder)
    build_dirs = {os.path.dirname(f) for f in files if BUILD_FILE_RE.search(f)}
    dock_dirs = {os.path.dirname(f) for f in files if DOCKERFILE_RE.search(f) and not NONPROD_DIR_RE.search(f)
                 and not PACKAGING_DIR_RE.search(os.path.dirname(f) + "/") and os.path.dirname(f) in build_dirs}
    m["svc_compose_built"], m["svc_k8s_workloads"], m["svc_dockerfile_dirs"] = compose_built, k8s, len(dock_dirs)
    m["n_services"] = max(1, compose_built, k8s, len(dock_dirs))
    m["_iac_set"] = set(iac_files)
    m["_ci_set"] = set(ci_files)
    return m

# ---------- effort (Robles et al. 2022) ----------
def fte_months(counts, theta_6m, months):
    """Developers at or above the threshold count as full-time; others count commits/threshold.
    The threshold is defined per 6 months and scaled to the window length."""
    th = theta_6m * months / 6.0
    return float(sum(min(1.0, n / th) for n in counts) * months)

def effort_for_period(human_commits, months, cfg):
    commits = defaultdict(int)
    days = defaultdict(set)
    for c in human_commits:
        commits[c["email"]] += 1
        days[c["email"]].add(c["date"].date())
    out = {"effort_pm": fte_months(commits.values(), cfg["THETA_6M"], months),
           "effort_pm_days": fte_months([len(v) for v in days.values()], cfg["THETA_6M"], months)}
    for t in cfg["THETA_ROBUST"]:
        out[f"effort_pm_t{t}"] = fte_months(commits.values(), t, months)
    return out

# ---------- one window ----------
def window_measures(commits, first_dates, tags, w_start, w_end, cfg):
    cw = [c for c in commits if w_start <= c["date"] < w_end]
    for c in cw:
        c["kind"] = author_kind(c["name"], c["email"])
    human = [c for c in cw if c["kind"] == "human"]
    row = {"commits_total": len(cw), "commits_human": len(human),
           "commits_bot": sum(c["kind"] == "bot" for c in cw),
           "commits_ai_author": sum(c["kind"] == "ai" for c in cw)}

    # effort: full-time-equivalent developer-months (Robles et al. 2022), main threshold + robustness variants
    row.update(effort_for_period(human, cfg["WINDOW_MONTHS"], cfg))

    dev_commits = defaultdict(int)
    for c in human:
        dev_commits[c["email"]] += 1
    row["team_size"] = len(dev_commits)
    row["core_devs"] = sum(1 for n in dev_commits.values() if n >= cfg["CORE_DEV_COMMITS"])

    # size: source lines added (added + modified lines appear as added in numstat)
    # bulk commits (imports, generated or vendored code) are left out of size
    src = test = bulk = nbulk = 0
    for c in human:
        cs = sum(a for a, d, p in c["files"] if file_class(p) == "src")
        if cs > cfg["BULK_COMMIT_LINES"]:
            bulk += cs
            nbulk += 1
            continue
        src += cs
        test += sum(a for a, d, p in c["files"] if file_class(p) == "test")
    row["ksloc"] = src / 1000.0
    row["ksloc_test"] = test / 1000.0
    row["ksloc_bulk_excluded"] = bulk / 1000.0
    row["n_bulk_commits"] = nbulk

    # AI commits (authored by agents or with an AI co-author trailer)
    nonbot = [c for c in cw if c["kind"] != "bot"]
    ai_commits = sum(1 for c in nonbot if c["kind"] == "ai" or has_ai_coauthor(c["body"]))
    row["ai_commit_share"] = ai_commits / len(nonbot) if nonbot else np.nan

    # releases
    months = (w_end - w_start).days / 30.44
    row["releases_per_month"] = sum(w_start <= t < w_end for t in tags) / months

    # team experience: commit-weighted mean months since each developer's first commit
    tenures, weights = [], []
    for e, n in dev_commits.items():
        fd = first_dates.get(e)
        if fd is not None:
            tenures.append(max(0.0, (w_start - fd).days / 30.44))
            weights.append(n)
    row["tenure_months"] = float(np.average(tenures, weights=weights)) if tenures else np.nan

    # company-led signal: share of human commits from the most common corporate email domain
    PUBLIC = {"gmail.com", "users.noreply.github.com", "outlook.com", "hotmail.com", "yahoo.com",
              "qq.com", "163.com", "protonmail.com", "proton.me", "icloud.com", "me.com", "live.com",
              "googlemail.com", "gmx.de", "web.de", "mail.ru", "yandex.ru", "126.com", "foxmail.com"}
    doms = defaultdict(int)
    for c in human:
        d = c["email"].split("@")[-1]
        if d and d not in PUBLIC and "." in d:
            doms[d] += 1
    top = max(doms.items(), key=lambda kv: kv[1]) if doms else (None, 0)
    row["top_corp_domain"] = top[0]
    row["top_corp_domain_share"] = top[1] / len(human) if human else np.nan
    row["_human_commits"] = human
    return row

## 4. API helpers

In [ ]:
# ===================== GITHUB / SCORECARD / AIDEV HELPERS =====================
import time, random, requests

SESSION = requests.Session()

def gh_get(url, params=None, search=False):
    """GET from the GitHub API with automatic waiting on rate limits."""
    if not url.startswith("http"):
        url = "https://api.github.com" + url
    headers = {"Authorization": f"Bearer {GITHUB_TOKEN}", "Accept": "application/vnd.github+json",
               "X-GitHub-Api-Version": "2022-11-28"}
    for attempt in range(6):
        if search:
            time.sleep(2.2)              # search API: 30 requests per minute
        r = SESSION.get(url, headers=headers, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (403, 429) and (r.headers.get("X-RateLimit-Remaining") == "0"
                                            or "rate limit" in r.text.lower()):
            reset = int(r.headers.get("X-RateLimit-Reset", time.time() + 60))
            wait = max(5, reset - time.time() + 5)
            print(f"  rate limit reached, waiting {wait/60:.1f} min")
            time.sleep(wait)
            continue
        if r.status_code in (502, 503, 504):
            time.sleep(10 * (attempt + 1))
            continue
        if r.status_code in (404, 409, 410, 422, 451):
            return None
        time.sleep(5)
    return None

def link_count(path, params=None):
    """Count items behind a paged GitHub list endpoint with one request (per_page=1, Link header)."""
    headers = {"Authorization": f"Bearer {GITHUB_TOKEN}", "Accept": "application/vnd.github+json"}
    p = dict(params or {}, per_page=1)
    for attempt in range(3):
        try:
            r = SESSION.get("https://api.github.com" + path, headers=headers, params=p, timeout=60)
        except Exception:
            time.sleep(5)
            continue
        if r.status_code in (403, 429) and r.headers.get("X-RateLimit-Remaining") == "0":
            time.sleep(max(5, int(r.headers.get("X-RateLimit-Reset", time.time() + 60)) - time.time() + 5))
            continue
        if r.status_code != 200:
            return 0
        m = re.search(r'[?&]page=(\d+)>; rel="last"', r.headers.get("Link", ""))
        return int(m.group(1)) if m else len(r.json())
    return 0

def commit_count_since(full_name, since):
    return link_count(f"/repos/{full_name}/commits", {"since": since})

def contributor_count(full_name):
    return link_count(f"/repos/{full_name}/contributors", {"anon": "false"})

def repo_meta(full_name):
    j = gh_get(f"/repos/{full_name}")
    if not j:
        return None
    return {"full_name": j["full_name"], "stars": j["stargazers_count"], "language": j["language"],
            "created_at": j["created_at"], "fork": j["fork"], "archived": j["archived"],
            "size_mb": j["size"] / 1024, "owner_type": j["owner"]["type"],
            "default_branch": j["default_branch"]}

def merged_pr_count(full_name, a, b):
    q = f"repo:{full_name} is:pr is:merged merged:{a:%Y-%m-%d}..{(b - pd.Timedelta(days=1)):%Y-%m-%d}"
    j = gh_get("/search/issues", params={"q": q, "per_page": 1}, search=True)
    return j["total_count"] if j else np.nan

def ci_run_count(full_name, a, b):
    j = gh_get(f"/repos/{full_name}/actions/runs",
               params={"created": f"{a:%Y-%m-%d}..{(b - pd.Timedelta(days=1)):%Y-%m-%d}", "per_page": 1})
    return j.get("total_count", np.nan) if j else np.nan

SCORECARD_CHECKS = ["Branch-Protection", "Code-Review", "SAST", "Dependency-Update-Tool", "Security-Policy"]

def scorecard(full_name):
    """Current OpenSSF Scorecard check scores (0-10); NaN if not available."""
    try:
        r = SESSION.get(f"https://api.securityscorecards.dev/projects/github.com/{full_name}", timeout=60)
        if r.status_code != 200:
            return {}
        checks = {c["name"]: c.get("score", -1) for c in r.json().get("checks", [])}
        return {f"sc_{k}": (checks[k] if checks.get(k, -1) >= 0 else np.nan) for k in SCORECARD_CHECKS}
    except Exception:
        return {}

# ---------- AIDev ----------
def load_aidev(version="v5"):
    base = f"hf://datasets/hao-li/AIDev@{version}"
    prs = pd.read_parquet(f"{base}/all_pull_request.parquet",
                          columns=["repo_url", "agent", "created_at", "merged_at"])
    prs["repo"] = prs["repo_url"].str.extract(r"github\.com/(?:repos/)?([^/]+/[^/]+?)(?:\.git)?/?$")[0]
    prs["created_at"] = pd.to_datetime(prs["created_at"], utc=True, errors="coerce")
    prs["merged_at"] = pd.to_datetime(prs["merged_at"], utc=True, errors="coerce")
    try:
        repos = pd.read_parquet(f"{base}/all_repository.parquet")
    except Exception as e:
        print("Could not load all_repository:", e)
        repos = None
    return prs, repos

def detect_col(df, *keys):
    for k in keys:
        for c in df.columns:
            if k in c.lower():
                return c
    return None

In [ ]:
rl = gh_get('/rate_limit')
print('GitHub API OK, requests left this hour:', rl['resources']['core']['remaining'] if rl else 'TOKEN PROBLEM')

## 5. Per-repository processing

In [ ]:
# ===================== PROCESS ONE REPOSITORY =====================
def process_repo(full_name, group, cfg, aidev_prs, aidev_cutoff, use_api=True):
    out_csv = os.path.join(cfg["OUT_DIR"], "repos", full_name.replace("/", "__") + ".csv")
    if os.path.exists(out_csv):
        print(f"  already done: {full_name}")
        return pd.read_csv(out_csv)
    meta = repo_meta(full_name) if use_api else {}
    sc = scorecard(full_name) if use_api else {}
    tmp = os.path.join(cfg["CLONE_DIR"], full_name.replace("/", "__"))
    t0 = time.time()
    clone_repo(full_name, tmp)
    try:
        windows = make_windows(cfg["START"], cfg["END"], cfg["WINDOW_MONTHS"])
        commits = read_commits(tmp, cfg["START"], cfg["END"])
        firsts = first_commit_dates(tmp)
        tags = tag_dates(tmp)
        agent = aidev_prs[(aidev_prs["repo"] == full_name) & aidev_prs["merged_at"].notna()] \
            if aidev_prs is not None else None
        rows = []
        for label, a, b in windows:
            sha = commit_at(tmp, b)
            if sha is None:
                continue
            r = {"repo": full_name, "group": group, "window": label,
                 "window_start": a.date().isoformat(), "end_commit": sha}
            r.update(window_measures(commits, firsts, tags, a, b, cfg))
            human = r.pop("_human_commits")
            if r["commits_total"] == 0:
                continue
            t = tree_measures(tmp, sha)
            iac_set = t.pop("_iac_set")
            ci_set = t.pop("_ci_set")
            r.update(t)
            r["iac_change_share"] = (sum(1 for c in human if any(p in iac_set for _, _, p in c["files"]))
                                     / len(human)) if human else np.nan
            r["ci_change_share"] = (sum(1 for c in human if any(p in ci_set for _, _, p in c["files"]))
                                    / len(human)) if human else np.nan
            # API-based window measures
            if use_api:
                r["merged_prs"] = merged_pr_count(full_name, a, b)
                r["ci_runs"] = ci_run_count(full_name, a, b)
            else:
                r["merged_prs"] = r["ci_runs"] = np.nan
            weeks = (b - a).days / 7
            r["ci_runs_per_week"] = r["ci_runs"] / weeks if not pd.isna(r["ci_runs"]) else np.nan
            if agent is not None and a < aidev_cutoff:
                n_agent = int(((agent["merged_at"] >= a) & (agent["merged_at"] < b)).sum())
                r["agent_merged_prs"] = n_agent
                r["agent_pr_share"] = (n_agent / r["merged_prs"]
                                       if r["merged_prs"] and not pd.isna(r["merged_prs"]) else np.nan)
            else:
                r["agent_merged_prs"] = r["agent_pr_share"] = np.nan   # window after AIDev coverage ends
            r.update(score_window(r, cfg))
            r.update({k: v for k, v in (meta or {}).items() if k in ("stars", "language", "created_at",
                                                                     "owner_type", "size_mb")})
            r.update(sc)
            r["include"] = int(r["core_devs"] >= cfg["MIN_CORE_DEVS"] and r["ksloc"] >= cfg["MIN_KSLOC"])
            rows.append(r)
        df = pd.DataFrame(rows)
        # robustness: effort over 6-month windows (the window length validated by Robles et al.)
        six = []
        for label, a, b in make_windows(cfg["START"], cfg["END"], 6):
            hc = [c for c in commits if a <= c["date"] < b and author_kind(c["name"], c["email"]) == "human"]
            if hc:
                e = effort_for_period(hc, 6, cfg)
                e.update(repo=full_name, half=label, commits_human=len(hc),
                         ksloc=sum(sum(x for x, _, p in c["files"] if file_class(p) == "src") for c in hc
                                   if sum(x for x, _, p in c["files"] if file_class(p) == "src")
                                   <= cfg["BULK_COMMIT_LINES"]) / 1000.0)
                six.append(e)
        os.makedirs(os.path.dirname(out_csv), exist_ok=True)
        os.makedirs(os.path.join(cfg["OUT_DIR"], "repos_6m"), exist_ok=True)
        pd.DataFrame(six).to_csv(out_csv.replace("/repos/", "/repos_6m/"), index=False)
        df.to_csv(out_csv, index=False)
        print(f"  done {full_name}: {len(df)} windows, {df['include'].sum() if len(df) else 0} included, "
              f"{(time.time() - t0)/60:.1f} min")
        return df
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

def score_window(r, cfg):
    """Turn raw indicators into 0-1 driver scores and COCOMO levels."""
    f = cfg["FULL_SCORE_AT"]
    s = {}
    # CI run counts are NOT used: the GitHub API keeps run history for about one year only
    s["DVOP"] = mean_avail([r["has_ci"], r["has_test_in_ci"], cap(r["n_ci_workflows"], f["n_ci_workflows"]),
                            cap(r["releases_per_month"], f["releases_per_month"])])
    s["AIAD"] = mean_avail([cap(r["agent_pr_share"], f["agent_pr_share"]),
                            cap(r["ai_commit_share"], f["ai_commit_share"]), r["has_ai_config"]])
    s["MSVC"] = min(1.0, math.log(r["n_services"]) / math.log(f["n_services"]))
    s["CLAU"] = mean_avail([int(r["n_iac_files"] > 0), cap(r["iac_change_share"], f["iac_change_share"]),
                            r["ci_deploy"]])
    s["MTEX"] = cap(r["tenure_months"], f["tenure_months"])
    s["SECU"] = mean_avail([r["has_security_policy"], r["has_dep_update"], r["has_sast"], r["has_codeowners"]])
    for k in list(s):
        s[f"{k}_level"] = to_level(s[k], cfg["LEVEL_CUTS"])
    return s

## 6. Load AIDev

In [ ]:
aidev_prs, aidev_repos = load_aidev(CFG["AIDEV_VERSION"])
AIDEV_CUTOFF = aidev_prs["created_at"].max()
print(f"{len(aidev_prs):,} agent PRs in {aidev_prs['repo'].nunique():,} repositories; data end: {AIDEV_CUTOFF:%Y-%m-%d}")

## 7. Choose the sample (first session only)
Selects the AI repositories and 2 comparison repositories for each, then saves the list. This takes 1–3 hours the first time; later sessions reuse the saved list.

In [ ]:
# ===================== CHOOSE THE PILOT REPOSITORIES =====================
def eligible(m, cfg):
    ok = (m is not None and not m["fork"] and not m["archived"]
            and m["stars"] >= cfg["MIN_STARS"] and m["language"] in cfg["LANGS"]
            and m["created_at"] < cfg["CREATED_BEFORE"] and m["size_mb"] <= cfg["MAX_REPO_MB"])
    if not ok:
        return False
    # activity screen: enough recent commits to have a real team (saves cloning inactive repos)
    m["recent_commits"] = commit_count_since(m["full_name"], cfg["ACTIVE_SINCE"])
    if m["recent_commits"] < cfg["MIN_RECENT_COMMITS"]:
        return False
    m["contributors"] = contributor_count(m["full_name"])
    return m["contributors"] >= cfg["MIN_CONTRIBUTORS"]

def select_ai_repos(prs, repos, cfg, n):
    merged = (prs[prs["merged_at"].notna() & prs["repo"].notna()]
              .groupby("repo").agg(n_agent_prs=("agent", "size"), first_agent_pr=("merged_at", "min"))
              .reset_index())
    cand = merged[(merged["n_agent_prs"] >= cfg["MIN_AGENT_PRS"]) &
                  (merged["first_agent_pr"] <= pd.Timestamp(cfg["ADOPTED_BY"], tz="UTC"))]
    # use AIDev's own star counts to pre-filter, if the column can be found
    if repos is not None:
        name_col = detect_col(repos, "full_name", "repo_url", "html_url", "url")
        star_col = detect_col(repos, "star")
        if name_col and star_col:
            rr = repos[[name_col, star_col]].copy()
            rr["repo"] = rr[name_col].astype(str).str.extract(
                r"(?:github\.com/(?:repos/)?)?([^/\s]+/[^/\s]+?)(?:\.git)?/?$")[0]
            tmp = cand.merge(rr[["repo", star_col]].rename(columns={star_col: "aidev_stars"}),
                             on="repo", how="left")
            if tmp["aidev_stars"].notna().mean() > 0.5:
                cand = tmp[tmp["aidev_stars"] >= cfg["MIN_STARS"]]
    print(f"AIDev repositories with >= {cfg['MIN_AGENT_PRS']} merged agent PRs, adopted by {cfg['ADOPTED_BY']}: {len(cand)}")
    cand = cand.sample(frac=1, random_state=cfg["SEED"])
    chosen, checked = [], 0
    for _, row in cand.iterrows():
        checked += 1
        m = repo_meta(row["repo"])
        if eligible(m, cfg):
            m.update(group="ai", first_agent_pr=row["first_agent_pr"], n_agent_prs=row["n_agent_prs"])
            chosen.append(m)
            print(f"  AI repo {len(chosen)}/{n}: {m['full_name']} ({m['language']}, {m['stars']} stars)")
        if len(chosen) >= n or checked >= cfg["MAX_CANDIDATES_CHECKED"]:
            break
    return chosen

def select_control_repos(ai_chosen, aidev_repos, cfg):
    """CONTROLS_PER_AI comparison repositories per AI repository: same language, similar stars,
    no AIDev activity."""
    used = set(aidev_repos) | {m["full_name"] for m in ai_chosen}
    chosen = []
    for i, m in enumerate(ai_chosen):
        lo, hi = max(cfg["MIN_STARS"], m["stars"] // 2), m["stars"] * 2
        q = (f'language:"{m["language"]}" stars:{lo}..{hi} created:<{cfg["CREATED_BEFORE"]} '
             f'pushed:>{cfg["ACTIVE_AFTER"]} fork:false archived:false')
        got = 0
        for page in (1, 2, 3):
            j = gh_get("/search/repositories", params={"q": q, "sort": "updated", "per_page": 50, "page": page},
                       search=True)
            items = (j or {}).get("items", [])
            for it in items:
                fn = it["full_name"]
                if fn in used or it["size"] / 1024 > cfg["MAX_REPO_MB"]:
                    continue
                c = repo_meta(fn)
                used.add(fn)
                if eligible(c, cfg):
                    c.update(group="control", matched_to=m["full_name"])
                    chosen.append(c)
                    got += 1
                    if got >= cfg["CONTROLS_PER_AI"]:
                        break
            if got >= cfg["CONTROLS_PER_AI"] or not items:
                break
        print(f"  [{i+1}/{len(ai_chosen)}] {m['full_name']}: {got} comparison repositories")
    return chosen

In [ ]:
sample_csv = CFG["OUT_DIR"] + "/sample.csv"
if os.path.exists(sample_csv):
    sample = pd.read_csv(sample_csv)
    print("Using saved sample")
else:
    ai = select_ai_repos(aidev_prs, aidev_repos, CFG, CFG["N_AI"])
    pd.DataFrame(ai).to_csv(CFG["OUT_DIR"] + "/sample_ai_only.csv", index=False)   # in case the next step is interrupted
    ctrl = select_control_repos(ai, aidev_prs["repo"].dropna().unique(), CFG)
    sample = pd.DataFrame(ai + ctrl).rename(columns={"full_name": "repo"})
    sample.to_csv(sample_csv, index=False)
print(sample["group"].value_counts().to_dict())
display(sample.groupby(["group", "language"]).size().unstack(0).fillna(0).astype(int))

## 8. Run (resumable)
Processes repositories not yet done, until `MAX_HOURS_PER_SESSION` is reached. Rerun the notebook to continue.

In [ ]:
import time
done = {f[:-4].replace("__", "/") for f in os.listdir(CFG["OUT_DIR"] + "/repos") if f.endswith(".csv")}
fail_csv = CFG["OUT_DIR"] + "/failures.csv"
todo = [r for r in sample.itertuples() if r.repo not in done]
print(f"Done: {len(done)} | to do: {len(todo)}")
t0 = time.time()
for i, row in enumerate(todo):
    if (time.time() - t0) / 3600 > CFG["MAX_HOURS_PER_SESSION"]:
        print("Session time budget reached. Download the results if not using Drive, then rerun to continue.")
        break
    print(f"[{len(done) + i + 1}/{len(sample)}] {row.repo} ({row.group})")
    try:
        process_repo(row.repo, row.group, CFG, aidev_prs, AIDEV_CUTOFF)
    except Exception as e:
        print("  FAILED:", str(e)[:300])
        pd.DataFrame([{"repo": row.repo, "error": str(e)[:300], "time": pd.Timestamp.now()}]).to_csv(
            fail_csv, mode="a", header=not os.path.exists(fail_csv), index=False)
left = len([r for r in sample.repo if not os.path.exists(CFG["OUT_DIR"] + "/repos/" + r.replace("/", "__") + ".csv")])
print("Repositories still to do:", left)

## 9. Combine (run when everything is done, or at any time for a progress check)

In [ ]:
import glob
files_ = glob.glob(CFG["OUT_DIR"] + "/repos/*.csv")
data = pd.concat([pd.read_csv(f) for f in files_], ignore_index=True)
sig = (data["agent_merged_prs"].fillna(0) > 0) | (data["ai_commit_share"].fillna(0) > 0) | (data["has_ai_config"] == 1)
first = data[sig].groupby("repo")["window_start"].min()
data["ai_adopt_start"] = data["repo"].map(first)
data["post_ai"] = (data["ai_adopt_start"].notna() & (data["window_start"] >= data["ai_adopt_start"])).astype(int)
owner = data["owner_type"] if "owner_type" in data else pd.Series("", index=data.index)
data["company_led"] = ((owner == "Organization") & (data["top_corp_domain_share"] >= 0.5)).astype(int)
data.to_csv(CFG["OUT_DIR"] + "/dataset_quarterly.csv", index=False)
six = pd.concat([pd.read_csv(f) for f in glob.glob(CFG["OUT_DIR"] + "/repos_6m/*.csv")], ignore_index=True)
six.to_csv(CFG["OUT_DIR"] + "/dataset_6month_effort.csv", index=False)
drivers = ["DVOP", "AIAD", "MSVC", "CLAU", "MTEX", "SECU"]
inc = data[data["include"] == 1]
print(f"{data['repo'].nunique()} repositories ({first.size} with an AI signal), {len(data)} windows, {len(inc)} included")
print("Never-adopting repositories:", data.groupby("repo")["ai_adopt_start"].first().isna().sum())
display(inc.groupby("group")[["effort_pm", "ksloc", "team_size"] + drivers].mean().round(2))
if os.path.exists(fail_csv):
    print("Failures so far:"); display(pd.read_csv(fail_csv).tail(20))

## 10. Download the results (needed without Google Drive; optional with it)

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/cocomo_full_run_results", "zip", CFG["OUT_DIR"])
files.download(zip_path)